# 配套实践 01-01：观察一条机器人轨迹

本练习对应基础篇第 01 章“机器人眼中的世界是什么”。我们将使用一段人工生成的桌面操作数据，观察观测、动作、状态转移、轨迹和时间窗口之间的关系。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/basics/01-robot-view-of-the-world/" target="_blank">在新标签页打开课程正文</a>

预计时间：10～15 分钟。不需要 GPU，也不需要下载外部数据。

## 使用方法

按照从上到下的顺序运行单元格。点击代码单元格左侧的运行按钮，或者按 Shift+Enter。练习中的数据只是为了解释概念，不代表真实机器人的运动精度。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
print("环境准备完成")

## 1. 构造一条简化轨迹

假设机械臂沿一个方向逐步接近杯子。每个时间步记录观测到的末端位置、执行的位移增量，以及动作执行后的新位置。

In [ ]:
time = np.arange(0, 0.7, 0.1)
observations = np.array([0.00, 0.04, 0.09, 0.15, 0.20, 0.24, 0.27])
actions = np.diff(observations)

trajectory = pd.DataFrame({
    "时间 t": time,
    "观测到的末端位置 o_t": observations,
})

trajectory["随后执行的动作 a_t"] = np.append(actions, np.nan)
trajectory

最后一个观测后没有动作，因为这条轨迹在该时刻结束。因此，这里有 7 个观测，但只有 6 个连接相邻观测的动作。

## 2. 查看状态转移

一次状态转移由当前观测、随后执行的动作和下一个观测组成。下面把整条轨迹改写成状态转移表。

In [ ]:
transitions = pd.DataFrame({
    "当前观测 o_t": observations[:-1],
    "动作 a_t": actions,
    "下一观测 o_(t+1)": observations[1:],
})

transitions["变化量"] = transitions["下一观测 o_(t+1)"] - transitions["当前观测 o_t"]
transitions

在这个简化例子中，动作正好等于位置变化量。真实机器人中，发出的控制命令与实际产生的运动通常不会完全相同。

## 3. 沿时间观察动作和观测

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

axes[0].plot(time, observations, marker="o")
axes[0].set_ylabel("position")
axes[0].set_title("Observation sequence")
axes[0].grid(alpha=0.3)

axes[1].step(time[:-1], actions, where="post")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("delta position")
axes[1].set_title("Action sequence")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

上图展示观测随时间的变化，下图展示每一步动作。动作位于两个相邻观测之间，而不是与最后一个观测一一对应。

## 4. 从轨迹中截取时间窗口

训练模型时经常只读取一段局部轨迹。修改下面的窗口起点和长度，观察窗口中观测数量与动作数量的关系。

In [ ]:
window_start = 1
action_horizon = 3

window_observations = observations[window_start:window_start + action_horizon + 1]
window_actions = actions[window_start:window_start + action_horizon]

print("窗口中的观测：", window_observations)
print("窗口中的动作：", window_actions)
print("观测数量：", len(window_observations))
print("动作数量：", len(window_actions))

## 5. 观察错误的时间对齐

下面故意把动作整体错开一个时间步。数组长度仍然合理，但动作不再解释对应的观测变化。这说明形状正确并不能保证时间语义正确。

In [ ]:
correct_error = np.mean(np.abs(actions - np.diff(observations)))
shifted_actions = np.roll(actions, 1)
shifted_error = np.mean(np.abs(shifted_actions - np.diff(observations)))

comparison = pd.DataFrame({
    "真实位置变化": np.diff(observations),
    "正确对齐动作": actions,
    "错开一步动作": shifted_actions,
})

display(comparison)
print(f"正确对齐的平均误差：{correct_error:.3f}")
print(f"错误对齐的平均误差：{shifted_error:.3f}")

## 练习

1. 将 action_horizon 改为 2、4 或 5，观察窗口长度。
2. 修改 observations 中某一个位置，观察动作与位置变化是否仍然一致。
3. 尝试把动作错开两个时间步，比较平均误差。

## 小结

一条轨迹由按时间排列的观测和动作组成。动作连接前后两个观测，因此观测数量通常比动作数量多一个。时间对齐错误不会必然改变数组形状，却会改变数据的真实含义。